# Silver · canais

**Origem:** `vcredit_bronze.canais` → **Destino:** `vcredit_silver.canais`

**Grão:** uma linha por canal. **Depende de:** bronze

Cadastro limpo dos canais de atendimento, usado como referência para o campo `canal` da tabela de chamados.

| Regra | Tratamento |
|---|---|
| `nome_canal` | Sem acento, minúsculo e sem espaços sobrando (mesmo padrão do `canal` em chamados) |
| `tipo_canal` e `status_canal` | Minúsculo e sem acento; valor fora do domínio vira nulo |
| ID não numérico ou nome vazio | Remove |
| ID repetido | Idênticos: mantém um; diferentes: remove todos |

**Decisões:** o padrão de texto é o mesmo do `canal` em chamados, para o join funcionar. Pipeline idempotente (lê da bronze e sobrescreve).

In [0]:
from pyspark.sql import functions as F, Window

catalog = "rocket_lab"
bronze_schema = f"{catalog}.vcredit_bronze"
silver_schema = f"{catalog}.vcredit_silver"

tabela_destino = f"{silver_schema}.canais"

COM = "áàâãäéèêëíìîïóòôõöúùûüç"
SEM = "aaaaaeeeeiiiiooooouuuuc"

# Função auxiliar utilizada para padronizar os textos (minúsculo, sem acento, sem espaços)
def normaliza(c):
    x = F.lower(F.trim(F.col(c)))
    x = F.translate(x, COM, SEM)
    x = F.regexp_replace(x, r"[\s\u00A0]+", " ")   # inclui espaço invisível 
    return F.lower(F.trim(x))

In [0]:
df = spark.table(f"{bronze_schema}.canais")
n_bronze = df.count()

# Padronizar as colunas
df = (df
    .withColumn("id_canal", F.trim("id_canal"))
    .withColumn("nome_canal", normaliza("nome_canal"))
    .withColumn("tipo_canal", F.lower(normaliza("tipo_canal")))
    .withColumn("status_canal", F.lower(normaliza("status_canal")))
)

# Regra 1: id_canal (só dígitos) e nome_canal são obrigatórios -> remove a linha
valido = (
    F.col("id_canal").rlike(r"^\d+$")
    & F.col("nome_canal").isNotNull()
    & (F.col("nome_canal") != "")
)
n_sem_chave = df.filter(~valido).count()
df = df.filter(valido)

# Regra 2: valor fora do domínio -> o campo vira nulo, a linha fica
tipo_ok = F.col("tipo_canal").isin("autosservico", "humano")
status_ok = F.col("status_canal").isin("ativo", "inativo")
n_dominio = df.filter(~tipo_ok | ~status_ok).count()
df = (df
    .withColumn("tipo_canal", F.when(tipo_ok, F.col("tipo_canal")))
    .withColumn("status_canal", F.when(status_ok, F.col("status_canal")))
)

# Regra 3) a): linhas duplicadas -> mantém uma só
cols_negocio = [c for c in df.columns if c not in ("ingest_timestamp", "source_file")] # seleciona todas as colunas menos a ingestion_timestamp e source_file.
n_antes = df.count()
df = df.dropDuplicates(cols_negocio)
n_identicas = n_antes - df.count() # conta quantas linhas foram removidas

# Regra 3) b): mesmo id_canal com dados diferentes -> anula todas
w = Window.partitionBy("id_canal")
df = df.withColumn("qtd", F.count("*").over(w))
n_conflitantes = df.filter("qtd > 1").count()
df = df.filter("qtd = 1").drop("qtd")

df = (df.withColumn("id_canal", F.col("id_canal").cast("int"))
      .select("id_canal", "nome_canal", "tipo_canal", "status_canal",
                "ingest_timestamp"))

# Idempotente (sempre lê da bronze e sobrescreve)
(df.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(tabela_destino))

# Resumo
n_silver = spark.table(tabela_destino).count()
print(f"bronze: {n_bronze} linhas")
print(f"  removidas por id/nome inválido:      {n_sem_chave}")
print(f"  linhas com valor fora do domínio:    {n_dominio} (campo anulado)")
print(f"  cópias idênticas removidas:          {n_identicas}")
print(f"  linhas anuladas por ID conflitante:  {n_conflitantes}")
print(f"silver: {n_silver} linhas")

display(spark.table(tabela_destino))